### **Tratamento de dados de matrícula**

In [1]:
import pandas as pd

# Seleção das colunas de interesse para o DataFrame final
colunas_interesse = [
    'Instituição', 'Unidade de Ensino', 'Nome de Curso', 
    'Tipo de Curso', 'Turno', 'Cor / Raça', 
    'Sexo', 'Idade', 'Faixa Etária', 'Renda Familiar', 
    'Situação de Matrícula', 'Categoria da Situação',
    'Código do Ciclo Matricula',
    'Data de Inicio do Ciclo', 'Data de Fim Previsto do Ciclo',
    'Vagas Regulares l1', 'Vagas Regulares l2', 'Vagas Regulares l5', 
    'Vagas Regulares l6', 'Vagas Regulares l9', 'Vagas Regulares AC'
]

**Carga dos dados**

In [2]:
chunks = pd.read_csv('../data/raw/microdados_matriculas_2023.csv', sep=';', encoding='utf-8', usecols=colunas_interesse, chunksize=50000)

tipos_graduacao = ['Licenciatura', 'Bacharelado', 'Tecnologia']

df_saida = pd.concat(
    [chunk[
        chunk['Instituição'].str.contains('BAHIA', na=False, case=False) &
        chunk['Tipo de Curso'].isin(tipos_graduacao)
    ] for chunk in chunks],
    ignore_index=True
)

# Remoção da coluna 'Instituição'
df_saida = df_saida.drop(columns=['Instituição'])

**Tratamento dos dados**

In [3]:
# Renomear colunas
df_saida.columns = [
    'categoria_situacao', 'cor_raca', 'codigo_ciclo_matricula',
    'data_fim_previsto_ciclo', 'data_inicio_ciclo', 'faixa_etaria',
    'idade', 'nome_curso', 'renda_familiar', 'sexo',
    'situacao_matricula', 'tipo_curso', 'turno', 'unidade_ensino',
    'vagas_regulares_ac', 'vagas_regulares_l1', 'vagas_regulares_l2',
    'vagas_regulares_l5', 'vagas_regulares_l6', 'vagas_regulares_l9'
]

In [4]:
# Conversão do formato das datas de string para datetime
df_saida['data_inicio_ciclo'] = pd.to_datetime(df_saida['data_inicio_ciclo'], errors='coerce', dayfirst=True)
df_saida['data_fim_previsto_ciclo'] = pd.to_datetime(df_saida['data_fim_previsto_ciclo'], errors='coerce', dayfirst=True)
print(df_saida[['data_inicio_ciclo', 'data_fim_previsto_ciclo']].dtypes)

data_inicio_ciclo          datetime64[us]
data_fim_previsto_ciclo    datetime64[us]
dtype: object


In [5]:
# Conversão do formato dos tipos de vagas de float para int
colunas_vagas = [
    'vagas_regulares_l1', 'vagas_regulares_l2', 'vagas_regulares_l5',
    'vagas_regulares_l6', 'vagas_regulares_l9', 'vagas_regulares_ac'
]
for col in colunas_vagas:
    df_saida[col] = pd.to_numeric(df_saida[col], errors='coerce').astype('Int64')

In [6]:
# Transformar a coluna 'codigo_ciclo_matricula' em string
df_saida['codigo_ciclo_matricula'] = df_saida['codigo_ciclo_matricula'].astype(str)

In [7]:
df_saida.info()

<class 'pandas.DataFrame'>
RangeIndex: 10034 entries, 0 to 10033
Data columns (total 20 columns):
 #   Column                   Non-Null Count  Dtype         
---  ------                   --------------  -----         
 0   categoria_situacao       10034 non-null  str           
 1   cor_raca                 10034 non-null  str           
 2   codigo_ciclo_matricula   10034 non-null  str           
 3   data_fim_previsto_ciclo  10034 non-null  datetime64[us]
 4   data_inicio_ciclo        10034 non-null  datetime64[us]
 5   faixa_etaria             10034 non-null  str           
 6   idade                    10034 non-null  int64         
 7   nome_curso               10034 non-null  str           
 8   renda_familiar           10034 non-null  str           
 9   sexo                     10034 non-null  str           
 10  situacao_matricula       10034 non-null  str           
 11  tipo_curso               10034 non-null  str           
 12  turno                    10034 non-null  st

In [ ]:
# Salvar DataFrame tratado em Parquet
# Optamos pelo formato Parquet para preservar os tipos de dados
df_saida.to_parquet('../data/processed/microdados_matriculas_tratados.parquet')